# P3 -- Exploratory CHM Analysis & ITC Delineation

This notebook demonstrates the core steps of the Individual Tree Crown (ITC)
delineation workflow:

1. Load and visualise a Canopy Height Model (CHM)
2. Detect treetops via variable-window local maxima
3. Segment crowns with marker-controlled watershed
4. Extract and display per-tree biometric metrics

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.utils.io import read_raster
from shared.data.generate_synthetic import generate_synthetic_chm, generate_synthetic_dtm

## 1. Generate / Load a CHM

In [ ]:
# Generate synthetic data for demonstration
DATA_DIR = Path("../data/scratch")
DATA_DIR.mkdir(parents=True, exist_ok=True)

chm_path = generate_synthetic_chm(DATA_DIR, n_trees=5)
dtm_path = generate_synthetic_dtm(DATA_DIR)

chm_data, chm_profile = read_raster(chm_path)
chm = chm_data[0]
print(f"CHM shape: {chm.shape}")
print(f"CHM range: [{chm.min():.2f}, {chm.max():.2f}] m")

## 2. Visualise CHM as a heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
transform = chm_profile["transform"]
extent = [
    transform.c,
    transform.c + chm.shape[1] * transform.a,
    transform.f + chm.shape[0] * transform.e,
    transform.f,
]
im = ax.imshow(chm, cmap="viridis", extent=extent, origin="upper")
ax.set_title("Canopy Height Model (m)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.colorbar(im, ax=ax, label="Height (m)")
plt.tight_layout()
plt.show()

## 3. Detect treetops

In [ ]:
from projects.p3_itc_delineation.src.treetops import detect_treetops

config = {
    "processing": {
        "crs": "EPSG:3310",
        "min_tree_height": 5.0,
    }
}

treetops = detect_treetops(chm_path, config)
print(f"Detected {len(treetops)} treetops")
treetops.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(chm, cmap="viridis", extent=extent, origin="upper")
treetops.plot(
    ax=ax, color="red", markersize=40, marker="^",
    label="Detected treetops"
)
ax.set_title("Treetop Detections over CHM")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Crown segmentation

In [ ]:
from projects.p3_itc_delineation.src.segmentation import segment_crowns

config_seg = {
    "processing": {
        "crs": "EPSG:3310",
        "min_tree_height": 5.0,
        "min_crown_area": 4.0,
    }
}

crowns = segment_crowns(chm_path, treetops, config_seg)
print(f"Segmented {len(crowns)} crowns")
crowns.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(chm, cmap="Greens", extent=extent, origin="upper", alpha=0.6)
crowns.boundary.plot(ax=ax, edgecolor="blue", linewidth=1.5, label="Crown boundaries")
treetops.plot(ax=ax, color="red", markersize=30, marker="^", label="Treetops")
ax.set_title("Crown Segmentation Results")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Per-tree biometric metrics

In [ ]:
from projects.p3_itc_delineation.src.metrics import extract_tree_metrics

metrics = extract_tree_metrics(crowns, chm_path, config)
print(f"Metrics computed for {len(metrics)} trees")

display_cols = [
    "tree_id", "crown_area_m2", "crown_diameter_m",
    "max_height_m", "mean_height_m", "dbh_inches", "stem_volume_cuft",
]
metrics[display_cols]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

if len(metrics) > 0:
    axes[0].bar(metrics["tree_id"], metrics["max_height_m"], color="forestgreen")
    axes[0].set_xlabel("Tree ID")
    axes[0].set_ylabel("Max Height (m)")
    axes[0].set_title("Tree Heights")

    axes[1].bar(metrics["tree_id"], metrics["dbh_inches"], color="sienna")
    axes[1].set_xlabel("Tree ID")
    axes[1].set_ylabel("DBH (inches)")
    axes[1].set_title("Estimated DBH")

    axes[2].bar(metrics["tree_id"], metrics["crown_area_m2"], color="steelblue")
    axes[2].set_xlabel("Tree ID")
    axes[2].set_ylabel("Crown Area (m\u00b2)")
    axes[2].set_title("Crown Areas")

plt.tight_layout()
plt.show()

## Summary

This exploratory notebook demonstrated the end-to-end ITC workflow on
synthetic data. The same pipeline functions are available as importable
modules and can be driven by the YAML configuration in
`configs/default.yaml` via `pipeline.run_pipeline()`.